In [4]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [5]:
tue_fri = pd.read_csv('TueToFri_IDS.csv')
monday= pd.read_csv('Mon_IDS.csv')
tue_fri.head()

,Protocol,Flow Duration,Total Fwd Packets,Fwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Bwd Packet Length Max,Bwd Packet Length Min,Flow Bytes/s,Flow Packets/s,...,Bwd Avg Packets/Bulk,Bwd Avg Bulk Rate,Init Fwd Win Bytes,Init Bwd Win Bytes,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Idle Std,Attack
0,6,113095465,48,9668,403,0,923,316,1.740123e+02,0.636630,...,0,0,571,2079,32,203985.50,575837.25,1629110,4277541.0,0
1,6,113473706,68,11364,403,0,1139,126,2.122254e+02,0.951762,...,0,0,390,2081,32,178326.88,503426.94,1424245,4229413.0,0
2,0,119945515,150,0,0,0,0,0,0.000000e+00,1.250568,...,0,0,-1,-1,0,6909777.50,11700000.00,20400000,24300000.0,0
3,6,60261928,9,2330,1093,0,1460,0,1.087088e+02,0.265508,...,0,0,8192,513,20,0.00,0.00,0,0.0,0
4,17,269,2,102,51,51,161,161,1.576208e+06,14869.888480,...,0,0,-1,-1,32,0.00,0.00,0,0.0,0


In [7]:
monday.head()

,Protocol,Flow Duration,Total Fwd Packets,Fwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Bwd Packet Length Max,Bwd Packet Length Min,Flow Bytes/s,Flow Packets/s,...,Bwd Avg Packets/Bulk,Bwd Avg Bulk Rate,Init Fwd Win Bytes,Init Bwd Win Bytes,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Idle Std,Attack
0,6,4,2,12,6,6,0,0,3.000000e+06,5.000000e+05,...,0,0,329,-1,20,0.0,0.0,0,0.0,0
1,6,1,2,12,6,6,0,0,1.200000e+07,2.000000e+06,...,0,0,329,-1,20,0.0,0.0,0,0.0,0
2,6,3,2,12,6,6,0,0,4.000000e+06,6.666667e+05,...,0,0,245,-1,20,0.0,0.0,0,0.0,0
3,6,1,2,12,6,6,0,0,1.200000e+07,2.000000e+06,...,0,0,245,-1,20,0.0,0.0,0,0.0,0
4,6,609,7,484,233,0,207,0,1.474548e+06,1.806240e+04,...,0,0,8192,2053,20,0.0,0.0,0,0.0,0


In [6]:
# Clean column spaces
monday.columns = monday.columns.str.strip()
tue_fri.columns = tue_fri.columns.str.strip()

In [9]:
# Remove duplicates
monday = monday.drop_duplicates()
tue_fri = tue_fri.drop_duplicates()

In [10]:

# Replace infinite values
monday.replace([np.inf, -np.inf], np.nan, inplace=True)
tue_fri.replace([np.inf, -np.inf], np.nan, inplace=True)


In [11]:

# Fill missing values
monday.fillna(0, inplace=True)
tue_fri.fillna(0, inplace=True)

In [16]:
X_rf = tue_fri.drop("Attack", axis=1)
y_rf = tue_fri["Attack"]

In [17]:
X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(
    X_rf, y_rf, test_size=0.2, stratify=y_rf, random_state=42
)

In [18]:
scaler = StandardScaler()

X_train_rf_scaled = scaler.fit_transform(X_train_rf)
X_test_rf_scaled = scaler.transform(X_test_rf)

joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']

In [19]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_rf_scaled, y_train_rf)

joblib.dump(rf, "rf_model.pkl")

['rf_model.pkl']

In [24]:
rf_pred = rf.predict(X_test_rf_scaled)

print("Random Forest Report:")
print(classification_report(y_test_rf, rf_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test_rf, rf_pred))

rf_probs = rf.predict_proba(X_test_rf_scaled)[:,1]
print("ROC-AUC:", roc_auc_score(y_test_rf, rf_probs))

Random Forest Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    215118
           1       1.00      1.00      1.00     65469

    accuracy                           1.00    280587
   macro avg       1.00      1.00      1.00    280587
weighted avg       1.00      1.00      1.00    280587

Confusion Matrix:
[[214961    157]
 [   210  65259]]
ROC-AUC: 0.9999719432096426


In [22]:
cv_scores = cross_val_score(
    rf, scaler.transform(X_rf), y_rf,
    cv=5, scoring="f1"
)

print("RF Cross-validation F1:", cv_scores.mean())

RF Cross-validation F1: 0.9574724940828772


In [41]:
X_iso = monday.drop("Attack", axis=1, errors="ignore")

scaler = joblib.load("scaler.pkl")
X_iso_scaled = scaler.transform(X_iso)

iso = IsolationForest(
    n_estimators=300,
    contamination=0.12,  # tune this
    random_state=42
)

iso.fit(X_iso_scaled)

joblib.dump(iso, "iso_model.pkl")

['iso_model.pkl']

In [43]:
iso_scores = iso.decision_function(X_test_rf_scaled)

# Lower score = more anomaly
threshold = np.percentile(iso_scores, 20)  # tune

iso_pred = np.where(iso_scores < threshold, 1, 0)

print("Isolation Forest Report:")
print(classification_report(y_test_rf, iso_pred))

Isolation Forest Report:
              precision    recall  f1-score   support

           0       0.86      0.90      0.88    215118
           1       0.60      0.52      0.56     65469

    accuracy                           0.81    280587
   macro avg       0.73      0.71      0.72    280587
weighted avg       0.80      0.81      0.80    280587



In [27]:
iso_norm = (iso_scores - iso_scores.min()) / (iso_scores.max() - iso_scores.min())

In [28]:
final_score = 0.7 * rf_probs + 0.3 * (1 - iso_norm)

final_pred = np.where(final_score > 0.5, 1, 0)

print("Hybrid Model Report:")
print(classification_report(y_test_rf, final_pred))

Hybrid Model Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    215118
           1       1.00      1.00      1.00     65469

    accuracy                           1.00    280587
   macro avg       1.00      1.00      1.00    280587
weighted avg       1.00      1.00      1.00    280587



In [29]:
def detect_intrusion(input_df):
    
    scaler = joblib.load("scaler.pkl")
    rf = joblib.load("rf_model.pkl")
    iso = joblib.load("iso_model.pkl")
    
    input_scaled = scaler.transform(input_df)
    
    rf_prob = rf.predict_proba(input_scaled)[:,1]
    iso_score = iso.decision_function(input_scaled)
    
    iso_norm = (iso_score - iso_score.min()) / (iso_score.max() - iso_score.min())
    
    final_score = 0.7 * rf_prob + 0.3 * (1 - iso_norm)
    
    prediction = np.where(final_score > 0.5, 1, 0)
    
    return prediction, final_score